# Seeing Machines — Screenshot Companion
**CompSci for Designers 2 — Final Project — Level 2 (The Companion)**

A multimodal RAG system over a personal Steam screenshot archive (470 images, 72 games).
Two independent retrieval routes over one corpus:

- **Level 1:** CLIP joint image–text embeddings — semantic search, exact game filtering, and
  reverse image identification.
- **Level 2:** Gemma 3 4B generates a structured JSON description of every screenshot;
  descriptions are embedded as text, retrieved independently, and a grounded natural-language
  answer is generated on top.

**Reproducibility:** all heavy artifacts (CLIP embeddings, captions, app-id→name mapping) are
cached in the GitHub repo this notebook clones. On a fresh runtime, `Runtime > Run all`
executes top-to-bottom with **no GPU captioning work and no manual uploads** — the captioning
cell sees the cache and skips itself. The GPU is only needed to *load* Gemma for live
grounded answers in the interface.

**Requirements to run:** a HuggingFace account with the (free) Gemma license accepted at
huggingface.co/google/gemma-3-4b-it, and a HF access token for the login cell.
Runtime: T4 GPU (`Runtime > Change runtime type`).

## 0. Setup — clone repo, resolve paths, sanity-check caches

In [ ]:
import os

REPO_URL = "https://github.com/Buendiajosemaria/SeeingMachines.git"
REPO_DIR = "/content/SeeingMachines"

if not os.path.isdir(REPO_DIR):
    print("Cloning repo...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("Repo already present. Pulling latest...")
    os.system(f"git -C {REPO_DIR} pull --ff-only")

# The repo mirrors the original MyDrive/SeeingMachines/ layout (nested folder).
ROOT_DIR  = f"{REPO_DIR}/SeeingMachines/screenshots"
CACHE_DIR = f"{REPO_DIR}/SeeingMachines/cache"

print("ROOT_DIR: ", ROOT_DIR,  "| exists:", os.path.isdir(ROOT_DIR))
print("CACHE_DIR:", CACHE_DIR, "| exists:", os.path.isdir(CACHE_DIR))

In [ ]:
!pip install -q open_clip_torch pillow pandas tqdm gradio
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# All imports for the whole notebook live here.
import json
import re
import difflib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import open_clip
import gradio as gr
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# Cold-start sanity check: every cached artifact this notebook expects.
# If anything is missing, this cell says so loudly BEFORE a later cell
# fails with a confusing traceback.
EXPECTED_CACHE = {
    "appid_to_name.json":          "app id -> game name mapping",
    "clip_embeddings.npy":         "Level 1 CLIP image embeddings",
    "clip_embeddings_index.csv":   "row index for the CLIP embeddings",
    "captions.json":               "Level 2 VLM descriptions (Gemma 3 4B)",
}

missing = []
for fname, desc in EXPECTED_CACHE.items():
    p = Path(CACHE_DIR) / fname
    status = "OK    " if p.exists() else "MISSING"
    print(f"[{status}] {fname:<28} {desc}")
    if not p.exists():
        missing.append(fname)

if missing:
    print(f"\n\u26a0\ufe0f  {len(missing)} cache file(s) missing: {missing}")
    print("The notebook can regenerate them, but that costs GPU time "
          "(captioning ~470 images takes 1.5-2.5h on a T4).")
else:
    print("\nAll caches present \u2014 this run needs no GPU captioning work.")

## 1. Scan the corpus
Walks `ROOT/<appid>/...` and records every image under each numeric app-id folder.
The app-id folder name is **ground-truth metadata** (from Steam's own screenshot
folder convention), resolved to a real game name from a static cached mapping —
no live API calls (grader-reproducibility requirement).

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def scan_corpus(root_dir: str) -> pd.DataFrame:
    rows = []
    for appid_dir in sorted(Path(root_dir).iterdir()):
        if not appid_dir.is_dir() or not re.fullmatch(r"\d+", appid_dir.name):
            continue
        for f in appid_dir.rglob("*"):
            if f.is_file() and f.suffix.lower() in IMG_EXTS:
                rows.append({"appid": appid_dir.name, "filepath": str(f), "filename": f.name})
    return pd.DataFrame(rows).drop_duplicates(subset="filepath").reset_index(drop=True)

corpus_df = scan_corpus(ROOT_DIR)
print(f"Found {len(corpus_df)} images across {corpus_df['appid'].nunique()} app-id folders.")

# Note: at 470 images this corpus exceeds the brief's 50-150 target for L1/L2
# (300 for L3). Kept deliberately: the size gives the retrieval comparison and
# the (planned) evaluation layer more distractors to work against, and all
# captioning cost has already been paid and cached. See documentation.
corpus_df.head()

In [ ]:
RESOLVED_CACHE = Path(CACHE_DIR) / "appid_to_name.json"

with open(RESOLVED_CACHE) as f:
    appid_to_name = json.load(f)
print(f"Loaded {len(appid_to_name)} app id -> name pairs.")

corpus_df["game_name"] = corpus_df["appid"].map(appid_to_name)

unmapped = corpus_df[corpus_df["game_name"].isna()]
if len(unmapped):
    print(f"\u26a0\ufe0f  {unmapped['appid'].nunique()} appid(s) missing from the mapping: "
          f"{sorted(unmapped['appid'].unique(), key=int)}")

print("\nImages per game:")
print(corpus_df.groupby("game_name").size().sort_values(ascending=False))

## 2. Level 1 — CLIP joint embedding space
Every image is embedded once with OpenCLIP ViT-B/32 into a joint image–text space;
text queries embed into the same space and cosine similarity ranks the corpus.
Embeddings load from cache; recomputation only happens if the cache is absent.

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(device).eval()

In [ ]:
EMB_PATH = Path(CACHE_DIR) / "clip_embeddings.npy"
EMB_INDEX_PATH = Path(CACHE_DIR) / "clip_embeddings_index.csv"

def embed_corpus(df: pd.DataFrame, force_recompute: bool = False):
    if EMB_PATH.exists() and EMB_INDEX_PATH.exists() and not force_recompute:
        print("Loading cached CLIP embeddings...")
        return np.load(EMB_PATH), pd.read_csv(EMB_INDEX_PATH)
    feats = []
    with torch.no_grad():
        for fp in tqdm(df["filepath"], desc="Embedding images"):
            img = clip_preprocess(Image.open(fp).convert("RGB")).unsqueeze(0).to(device)
            feat = clip_model.encode_image(img)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            feats.append(feat.cpu().numpy()[0])
    feats = np.stack(feats)
    np.save(EMB_PATH, feats)
    df.to_csv(EMB_INDEX_PATH, index=False)
    return feats, df

clip_embeddings, indexed_df = embed_corpus(corpus_df)

# The cached index may carry filepaths from wherever it was originally built
# (e.g. a Drive mount). Rebase them onto the current ROOT_DIR so image loading
# works regardless of where this session cloned the repo.
_old = indexed_df["filepath"].iloc[0]
_old_prefix = _old[:_old.index("/screenshots/") + len("/screenshots")]
if _old_prefix != ROOT_DIR:
    print(f"Rebasing cached filepaths: {_old_prefix} -> {ROOT_DIR}")
    indexed_df["filepath"] = indexed_df["filepath"].str.replace(_old_prefix, ROOT_DIR, n=1, regex=False)

print(f"{len(indexed_df)} images indexed, embedding matrix {clip_embeddings.shape}.")

### Level 1 retrieval logic
- `embed_text` + `cosine_topk`: semantic queries ("a nighttime city street").
- `find_named_game` + exact filter: "pull up all screenshots from \<game\>" — a literal
  metadata lookup, since the folder structure already guarantees the answer.
- `embed_query_image`: reverse identification of an unsorted screenshot by
  nearest-neighbor against the indexed corpus, reported with a similarity score.

In [ ]:
def embed_text(query: str) -> np.ndarray:
    with torch.no_grad():
        tok = clip_tokenizer([query]).to(device)
        feat = clip_model.encode_text(tok)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

def embed_query_image(pil_img: Image.Image) -> np.ndarray:
    with torch.no_grad():
        img = clip_preprocess(pil_img.convert("RGB")).unsqueeze(0).to(device)
        feat = clip_model.encode_image(img)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

def cosine_topk(query_vec: np.ndarray, top_k: int = 8, min_sim: float = 0.0) -> pd.DataFrame:
    sims = clip_embeddings @ query_vec
    order = np.argsort(-sims)[:top_k]
    results = indexed_df.iloc[order].copy()
    results["similarity"] = sims[order]
    return results[results["similarity"] >= min_sim]

def find_named_game(query: str):
    names = indexed_df["game_name"].dropna().unique().tolist()
    lowered = [n.lower() for n in names]
    match = difflib.get_close_matches(query.lower(), lowered, n=1, cutoff=0.6)
    if match:
        return names[lowered.index(match[0])]
    for n in names:
        if n.lower() in query.lower() or query.lower() in n.lower():
            return n
    return None

def search(query_text: str = None, query_image: Image.Image = None,
           top_k: int = 8, min_sim: float = 0.0):
    """Level 1 search: pure CLIP image-embedding similarity + metadata filtering."""
    if query_image is not None:
        vec = embed_query_image(query_image)
        results = cosine_topk(vec, top_k=top_k, min_sim=min_sim)
        if len(results):
            best = results.iloc[0]
            answer = (f"Best guess: **{best['game_name']}** "
                      f"(similarity {best['similarity']:.2f}, appid {best['appid']})")
        else:
            answer = "No confident match found."
        return answer, results

    if query_text:
        named = find_named_game(query_text)
        wants_all = any(w in query_text.lower() for w in ["all", "every", "pull up", "show me"])
        if named and wants_all:
            results = indexed_df[indexed_df["game_name"] == named].copy()
            results["similarity"] = 1.0
            return f"Showing all {len(results)} screenshots tagged **{named}** (exact metadata match).", results
        vec = embed_text(query_text)
        results = cosine_topk(vec, top_k=top_k, min_sim=min_sim)
        return f'Top {len(results)} semantic matches for: "{query_text}" (Level 1, CLIP)', results

    return "Enter a text query or upload an image.", indexed_df.iloc[0:0]

## 3. Level 2 — Gemma 3 4B: load model
Requires a HuggingFace token with the Gemma license accepted. The model is needed at
demo time for grounded answers, so this loads regardless of caption cache state.

Quantization decisions (all deliberate, all documented failures behind them):
- **4-bit NF4** — a T4 cannot fit the 4B model unquantized.
- **float16 compute** — float32 proved ~2\u00d7 slower during the first captioning run;
  bfloat16 is unsupported on T4 hardware.
- **`do_pan_and_scan=False`** (in the captioning call) — Gemma's default multi-crop
  tiling multiplies visual tokens ~4-6\u00d7 per image for no captioning benefit at
  this resolution.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig

VLM_ID = "google/gemma-3-4b-it"

vlm_quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

vlm_model = Gemma3ForConditionalGeneration.from_pretrained(
    VLM_ID,
    device_map="auto",
    quantization_config=vlm_quant_config,
).eval()
vlm_processor = AutoProcessor.from_pretrained(VLM_ID)

print(f"Loaded {VLM_ID} in 4-bit. Memory footprint: {vlm_model.get_memory_footprint() / 1e9:.2f} GB")

## 4. Level 2 — captioning (skips itself when cached)
The description schema is the central design decision of this level:

- `visible_elements` as an explicit object list is what lets Level 2 "see" small things
  a holistic CLIP embedding dilutes away (the "fish in the corner of the frame" problem).
- `guessed_genre_or_title` is the model's **blind** guess — it is never told the real game
  name. Comparing this guess against the App-ID-derived ground truth generates the
  mis-seeing dossier automatically.
- `visible_text` asks for a **short summary, not verbatim transcription** — the original
  verbatim instruction caused 324/470 captions to blow the token budget and truncate on
  text-dense screenshots (scoreboards, HUD readouts). Bounding the field fixed all but one;
  the survivor failed on curly-quote drift instead (see `misseeing_truncation_failures.json`
  and the documentation). Both failure classes are handled below: bounded prompt + quote
  normalization before parsing.

In [ ]:
CAPTION_SYSTEM_PROMPT = (
    "You analyze a single video game screenshot. Describe ONLY what is visibly present \u2014 "
    "do not use any outside knowledge of specific game titles. Respond with STRICT JSON only, "
    "no markdown fences, no commentary outside the JSON, matching exactly this schema:\n"
    '{"scene_type": "menu_ui|combat|exploration|cutscene_dialogue|inventory_crafting|map_screen|other", '
    '"visible_elements": ["short noun phrases for distinct objects, creatures, or items you can see"], '
    '"setting_description": "1-2 sentences describing the environment and any action", '
    '"visible_text": "a SHORT summary of on-screen text, max 20 words - do NOT transcribe everything", '
    '"guessed_genre_or_title": "your own blind guess at genre or game title, based only on visuals", '
    '"confidence_notes": "anything you are unsure about"}'
)

def resize_for_vlm(img: Image.Image, max_side: int = 512) -> Image.Image:
    w, h = img.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

def caption_image(pil_img: Image.Image, max_new_tokens: int = 300) -> dict:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": CAPTION_SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": pil_img},
            {"type": "text", "text": "Describe this screenshot."},
        ]},
    ]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
        processor_kwargs={"do_pan_and_scan": False},
    ).to(vlm_model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    raw = vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

    cleaned = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    cleaned = cleaned.replace("\u201c", '"').replace("\u201d", '"')  # curly-quote drift fix
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw_text": raw}

In [ ]:
CAPTION_CACHE = Path(CACHE_DIR) / "captions.json"
SAVE_EVERY = 10

def load_captions() -> dict:
    if CAPTION_CACHE.exists():
        with open(CAPTION_CACHE) as f:
            return json.load(f)
    return {}

def save_captions(d: dict):
    with open(CAPTION_CACHE, "w") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

captions = load_captions()
to_caption = [r for r in indexed_df.itertuples() if r.filepath not in captions]
print(f"{len(captions)} cached, {len(to_caption)} left to caption this run.")

for i, row in enumerate(tqdm(to_caption, desc="Captioning")):
    img = resize_for_vlm(Image.open(row.filepath).convert("RGB"))
    captions[row.filepath] = caption_image(img)
    if (i + 1) % SAVE_EVERY == 0 or i == len(to_caption) - 1:
        save_captions(captions)

parse_errors = [fp for fp, v in captions.items() if v.get("parse_error")]
print(f"Caption cache: {len(captions)} total, {len(parse_errors)} known parse failure(s) "
      f"(kept deliberately \u2014 handled via raw-text fallback and documented in the dossier).")

## 5. Level 2 — caption retrieval + grounded answers
Captions are flattened to one string per image and embedded with the **same CLIP text
encoder** as Level 1 (one model stack, deliberately). Retrieval over caption embeddings is
fully independent of Level 1's image embeddings — two routes over one corpus.

The grounded answer feeds retrieved captions (text, not images) back into Gemma. The
prompt explicitly asks for **nearest-related results instead of a flat "no match"** —
so "fish" against a fishless corpus surfaces water/underwater scenes with an explanation
rather than a dead end. Passing actual retrieved images into context is Level 3 territory.

In [ ]:
def caption_to_text(cap: dict) -> str:
    if cap.get("parse_error"):
        return cap.get("raw_text", "")  # fallback: still searchable, just unstructured
    elements = ", ".join(cap.get("visible_elements", []))
    return (f"{cap.get('setting_description', '')} "
            f"Visible elements: {elements}. "
            f"Scene type: {cap.get('scene_type', '')}. "
            f"On-screen text: {cap.get('visible_text', '')}").strip()

indexed_df["caption_text"] = indexed_df["filepath"].map(lambda fp: caption_to_text(captions.get(fp, {})))
indexed_df["guessed_genre_or_title"] = indexed_df["filepath"].map(
    lambda fp: captions.get(fp, {}).get("guessed_genre_or_title", ""))

CAPTION_EMB_PATH = Path(CACHE_DIR) / "caption_embeddings.npy"

def embed_captions(df: pd.DataFrame, force_recompute: bool = False) -> np.ndarray:
    if CAPTION_EMB_PATH.exists() and not force_recompute:
        embs = np.load(CAPTION_EMB_PATH)
        if len(embs) == len(df):
            print("Loading cached caption embeddings...")
            return embs
        print("Cached caption embeddings don't match corpus size \u2014 recomputing.")
    embs = []
    with torch.no_grad():
        for text in tqdm(df["caption_text"], desc="Embedding captions"):
            tok = clip_tokenizer([text or " "]).to(device)
            feat = clip_model.encode_text(tok)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            embs.append(feat.cpu().numpy()[0])
    embs = np.stack(embs)
    np.save(CAPTION_EMB_PATH, embs)
    return embs

caption_embeddings = embed_captions(indexed_df)

def search_captions(query_text: str, top_k: int = 8, min_sim: float = 0.0) -> pd.DataFrame:
    """Level 2 search: caption-text embedding similarity."""
    vec = embed_text(query_text)
    sims = caption_embeddings @ vec
    order = np.argsort(-sims)[:top_k]
    results = indexed_df.iloc[order].copy()
    results["similarity"] = sims[order]
    return results[results["similarity"] >= min_sim]

In [ ]:
def generate_grounded_answer(query: str, retrieved_rows: pd.DataFrame, max_new_tokens: int = 200) -> str:
    if len(retrieved_rows) == 0:
        return "No matching screenshots found in the archive for this query."
    context = "\n".join(f"- [{row.game_name}] {row.caption_text}"
                         for row in retrieved_rows.itertuples())
    messages = [
        {"role": "system", "content": [{"type": "text", "text":
            "Answer the question using ONLY the screenshot descriptions provided. "
            "If nothing matches the query directly, do NOT just say there is no match: "
            "instead, point out the most closely related screenshots and explain the connection "
            "(e.g. for 'fish', underwater or water scenes are relevant). Be brief and concrete."}]},
        {"role": "user", "content": [{"type": "text", "text":
            f"Screenshot descriptions:\n{context}\n\nQuestion: {query}"}]},
    ]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(vlm_model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

## 6. Level 1 vs Level 2 — required comparison (\u22655 shared queries)
Same queries through both routes. "fish" is here on purpose: no dedicated fish screenshots
exist in this corpus, making it the clearest probe of how each route handles absence —
CLIP's holistic image similarity vs. caption vocabulary matching. Interpretation of the
divergence lives in the project documentation.

In [ ]:
comparison_queries = [
    "fish",
    "a nighttime city street",
    "a health bar or HUD element",
    "an inventory or crafting menu",
    "a boss fight",
]

for q in comparison_queries:
    print(f"\n=== Query: {q!r} ===")
    l1 = cosine_topk(embed_text(q), top_k=5)
    l2 = search_captions(q, top_k=5)
    print("Level 1 (CLIP image embeddings):")
    print(l1[["game_name", "filename", "similarity"]].to_string(index=False))
    print("Level 2 (caption embeddings):")
    print(l2[["game_name", "filename", "similarity"]].to_string(index=False))

## 7. Gradio interface
Functional, not polished (per the brief). The route selector makes the L1/L2 comparison
tangible: two labeled galleries, side by side in "Both" mode.

In [ ]:
def to_gallery(results):
    return [(r["filepath"], f"{r['game_name']} \u00b7 {r.get('similarity', 0):.2f}")
            for _, r in results.iterrows()]

def gradio_search(query_text, query_image_path, route):
    empty = []
    if query_image_path:
        pil_img = Image.open(query_image_path).convert("RGB")
        answer, results = search(query_image=pil_img, top_k=12)
        return answer, to_gallery(results), empty

    if not query_text:
        return "Enter a text query or upload an image.", empty, empty

    if route == "Level 1 \u2014 CLIP image similarity":
        answer, results = search(query_text=query_text, top_k=12)
        return answer, to_gallery(results), empty

    if route == "Level 2 \u2014 caption-grounded":
        results = search_captions(query_text, top_k=8)
        answer = generate_grounded_answer(query_text, results)
        return answer, empty, to_gallery(results)

    # Both side-by-side
    l1_results = search(query_text=query_text, top_k=6)[1]
    l2_results = search_captions(query_text, top_k=6)
    grounded = generate_grounded_answer(query_text, l2_results)
    return f"**Level 2 grounded answer:** {grounded}", to_gallery(l1_results), to_gallery(l2_results)

with gr.Blocks(title="Seeing Machines \u2014 Screenshot Companion") as demo:
    gr.Markdown("# Seeing Machines: Screenshot Companion")
    with gr.Row():
        txt = gr.Textbox(label="Ask something", placeholder='e.g. "fish" or "a nighttime city street"')
        img = gr.Image(label="...or upload a screenshot to identify", type="filepath")
    route = gr.Radio(
        ["Level 1 \u2014 CLIP image similarity", "Level 2 \u2014 caption-grounded", "Both side-by-side"],
        value="Level 2 \u2014 caption-grounded", label="Retrieval route",
    )
    btn = gr.Button("Search")
    answer_box = gr.Markdown()
    with gr.Row():
        gallery_l1 = gr.Gallery(label="Level 1 \u2014 CLIP image similarity", columns=3, height=420)
        gallery_l2 = gr.Gallery(label="Level 2 \u2014 caption retrieval", columns=3, height=420)
    btn.click(gradio_search, inputs=[txt, img, route], outputs=[answer_box, gallery_l1, gallery_l2])

demo.launch(share=True)

## Appendix — maintenance utilities (not part of a normal run)
Run these only when you have produced new artifacts (e.g. after a re-caption) and need to
get them out of the session before it dies.

In [ ]:
# Download all cache artifacts to your machine.
# from google.colab import files
# for fname in ["captions.json", "clip_embeddings.npy", "clip_embeddings_index.csv",
#               "appid_to_name.json", "caption_embeddings.npy",
#               "misseeing_truncation_failures.json"]:
#     p = Path(CACHE_DIR) / fname
#     if p.exists():
#         files.download(str(p))

In [ ]:
# Push cache artifacts back to the GitHub repo.
# import subprocess, os
# GITHUB_TOKEN = "paste_token_here"  # needs 'repo' scope; never commit this cell with a real token
# def git(cmd):
#     r = subprocess.run(cmd, shell=True, cwd=REPO_DIR, capture_output=True, text=True)
#     print(r.stdout or r.stderr)
# git("git config user.email 'you@example.com'")
# git("git config user.name 'Your Name'")
# git(f"git add {os.path.relpath(CACHE_DIR, REPO_DIR)}")
# git("git commit -m 'Update cache artifacts'")
# git(f"git push https://{GITHUB_TOKEN}@github.com/Buendiajosemaria/SeeingMachines.git main")